# R2-04 · Fundamentos de ML — Profundización (opcional) 🔬

**Formación Pública — Rama R2 · Científico de Datos · Notebook de profundización**

Opcional y de **transferencia**: el mismo método (baseline → `fit`/`predict` → MAE en prueba) sobre
otro dato del Estado, la **matrícula escolar 2025** (MINEDUC). Dos lecciones nuevas: cuándo **sí** vale
un corte aleatorio (no hay tiempo de por medio) y cómo reconocer la **fuga de datos** cuando un modelo
sale «demasiado bueno».

> Cada ejercicio termina en una **celda de chequeo** con ✅ / pista.

In [ ]:
import pandas as pd, numpy as np, os, urllib.request
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
CSV = "matricula.csv"
if not os.path.exists(CSV):
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/{CSV}", CSV)
mat = pd.read_csv(CSV)    # matrícula 2025 por establecimiento (MINEDUC)
print("establecimientos:", mat.shape)
mat[["nombre", "rural", "mat_total", "mat_hombres", "mat_mujeres", "cursos"]].head()

## 1. Otro problema, el mismo método: ¿cuántos cursos tendrá un colegio?

Planificar dotación docente o infraestructura exige estimar **cursos** a partir de la **matrícula**. Aquí no hay tiempo de por medio (es una foto de 2025), así que un **corte aleatorio** sí es válido. Primero, el baseline: predecir siempre el promedio de cursos del entrenamiento.

### ✍️ Ejercicio 1 — Divide 70/30 al azar y mide el baseline

In [ ]:
X = mat[["mat_total"]]
y = mat["cursos"]
# TODO: train_test_split(X, y, test_size=0.3, random_state=42)
X_train, X_test, y_train, y_test = ..., ..., ..., ...
# TODO: MAE de predecir siempre y_train.mean()
mae_base = ...
print("MAE baseline:", mae_base)

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    assert len(X_train) + len(X_test) == len(mat)
    assert abs(len(X_test) / len(mat) - 0.3) < 0.01
    assert abs(mae_base - mean_absolute_error(y_test, np.full(len(y_test), y_train.mean()))) < 1e-9
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'train_test_split(X, y, test_size=0.3, random_state=42); np.full(len(y_test), y_train.mean()) es la predicción constante.')
    print("   Detalle:", e)

## 2. La regresión y su lectura

Entrena la regresión lineal y traduce su coeficiente a lenguaje de gestión: **cursos por cada 100 estudiantes**.

### ✍️ Ejercicio 2 — Entrena, evalúa e interpreta el coeficiente

In [ ]:
# TODO: entrena LinearRegression con (X_train, y_train)
modelo = ...
mae_lr = mean_absolute_error(y_test, modelo.predict(X_test))
# TODO: 100 * el coeficiente de mat_total (modelo.coef_[0])
cursos_por_100 = ...
print("MAE:", mae_lr, "| cursos por 100 estudiantes:", cursos_por_100)

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    assert mae_lr < mae_base / 2, "El modelo debería reducir el error a menos de la mitad"
    assert 1 < cursos_por_100 < 5
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'LinearRegression().fit(X_train, y_train); cursos_por_100 = 100 * modelo.coef_[0].')
    print("   Detalle:", e)

## 3. La trampa: un modelo «perfecto» por **fuga de datos**

Ahora predice `mat_total` usando `mat_hombres` y `mat_mujeres`. El resultado será casi perfecto… porque `mat_total` **es** la suma de ambas. Eso no es predecir: es copiar la respuesta. Cuando un modelo sale demasiado bueno, sospecha **fuga** (*leakage*): una feature que contiene el objetivo o que no tendrías al momento de predecir.

### ✍️ Ejercicio 3 — Mide el R² del modelo con fuga

In [ ]:
Xf = mat[["mat_hombres", "mat_mujeres"]]
Xf_tr, Xf_te, yf_tr, yf_te = train_test_split(Xf, mat["mat_total"], test_size=0.3, random_state=42)
m_fuga = LinearRegression().fit(Xf_tr, yf_tr)
# TODO: r2_score entre yf_te y las predicciones de m_fuga sobre Xf_te
r2_fuga = ...
print("R² con fuga:", r2_fuga)

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    assert r2_fuga > 0.999, "Con la suma como feature, el R² es prácticamente 1"
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'r2_score(yf_te, m_fuga.predict(Xf_te)).')
    print("   Detalle:", e)

## 4. El modelo honesto

Si de verdad necesitas estimar la matrícula de un colegio, usa información que **sí tendrías** antes: cuántos `cursos` tiene y si es `rural`. El R² baja… y eso está bien: ahora es real.

### ✍️ Ejercicio 4 — Predice la matrícula sin fuga y compara con su baseline

In [ ]:
Xh = mat[["cursos", "rural"]]
Xh_tr, Xh_te, yh_tr, yh_te = train_test_split(Xh, mat["mat_total"], test_size=0.3, random_state=42)
# TODO: entrena el modelo honesto y calcula su MAE en prueba
m_honesto = ...
mae_honesto = ...
mae_base_mat = mean_absolute_error(yh_te, np.full(len(yh_te), yh_tr.mean()))
r2_honesto = r2_score(yh_te, m_honesto.predict(Xh_te))
print("MAE honesto:", mae_honesto, "| baseline:", round(mae_base_mat, 1), "| R²:", r2_honesto)

In [ ]:
# ── Celda de chequeo · Ejercicio 4 ──────────────────────────────────────────
try:
    assert mae_honesto < mae_base_mat
    assert r2_honesto < r2_fuga, "El modelo honesto es peor que el tramposo… y es el único que sirve"
    print("✅ Ejercicio 4: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'LinearRegression().fit(Xh_tr, yh_tr) y mean_absolute_error(yh_te, m_honesto.predict(Xh_te)).')
    print("   Detalle:", e)

## Para llevar

- **Aleatorio o temporal:** depende de cómo se usará el modelo. Foto de un año → aleatorio; pronóstico → temporal.
- **Demasiado bueno para ser cierto** casi siempre es fuga: pregúntate si tendrías esa feature al momento de predecir.

---

*Fin de la profundización de R2-04.*